# 05 - XGBoost Model (Advanced Tree-Based Classifier)

In this notebook we build and tune an **XGBoost classifier** using the engineered Bank Marketing dataset form the silver layer.

Compared to Logistic Regression, XGBoost is better at:

- Capturing **nonlinear relationships** and complex interactions
- Handling **skewed features** and **outliers** more naturally
- Working with **mixed numeric/categorical** data (after encoding)
- Leveraging **class imbalance handling** and robust regularization


In [ ]:
# --- Basic Imports ---
from utils import *

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, roc_auc_score,
                             precision_recall_fscore_support, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, log_loss)

from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from category_encoders import WOEEncoder
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier, Booster
import xgboost as xgb
import shap
import json

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def find_best_threshold(y_true, y_prob, metric="f1"):
    thresholds = np.linspace(0.01, 0.99, 200)
    scores = []

    for t in thresholds:
        y_pred = (y_prob > t).astype(int)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="binary", zero_division=0
        )

        if metric == "precision":
            scores.append(precision)
        elif metric == "recall":
            scores.append(recall)
        else:  # default f1
            scores.append(f1)

    scores = np.array(scores)
    best_idx = scores.argmax()
    best_threshold = thresholds[best_idx]
    best_score = scores[best_idx]

    return best_threshold, best_score, thresholds, scores, metric

def replace_unknowns(df):
    return df.replace("unknown", np.nan)

In [ ]:
# Plot Themes
ACCENT_COLOR = "#2ab7ca"
SECOND_COLOR = "#0d3b66"
HIGHLIGHT_COLOR = "#ff0000ea"

# --- Theme & custom accent color ---
sns.set_theme(style="whitegrid", palette="mako")

model_type = 'XGBoost_Model'
start_date = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
timestamp = dt.datetime.strptime(start_date, '%Y%m%d_%H%M%S').strftime("%Y-%m-%d %H:%M:%S")
path = MODEL_DIR+model_type+"_"+start_date

os.makedirs(path, exist_ok=True)

## 1. Feature Engineering

We start from the cleaned **Silver-layer** dataset (`base_dataset.csv`) and engineer features tailored to tree-based models like XGBoost.

Although trees are robust to raw, skewed data, good feature engineering can still help:

- Improve performance  
- Enhance interpretability  
- Align the model with business patterns (segments, seasonality, interactions)

In [ ]:
df = pd.read_csv(os.path.join(SILVER_PATH, 'base_dataset.csv'), sep=';')

### 1.1 Job Grouping

As with the Logistic Regression notebook, we reduce the granularity of `job`:

- Group similar occupations into broader categories (e.g., services, self-employed, no labor force)
- Preserve meaningful distinctions (e.g., admin, management, technician)

This reduces noise and stabilizes category-level patterns without losing key information.

In [ ]:
# Show job categories from dataset
job_counts = df['job'].value_counts(normalize=True)
print(job_counts)

# Map jobs to categories to reduce cardinality
job_mapping = {
    "admin.": "admin",
    "blue-collar": "blue_collar",
    "technician": "technician",
    "services": "services_group",
    "housemaid": "services_group",
    "management": "management",
    "retired": "no_labor_force",
    "student": "no_labor_force",
    "unemployed": "no_labor_force",
    "entrepreneur": "self_employed_group",
    "self-employed": "self_employed_group",
    "unknown": "unknown"
}
df['job'] = df['job'].map(job_mapping)

### 1.2 Education Simplification

Similarly to what was done in the construction of the Linear Regression model, `education` variable contains multiple related levels (e.g. basic.4y, basic.6y, basic.9y).

Again, we consolidate them into higher-level buckets:

- `basic_education`
- `high_school`
- `university_degree`
- `professional_course`
- `illiterate`
- `unknown`

The new column `education_simplified`:

- Makes the feature easier to interpret
- Reduces sparsity after encoding
- Aligns better with typical business reporting categories

In [ ]:
# Show education categories from dataset
education_counts = df['education'].value_counts(normalize=True)
print(education_counts)

# ------------------------------------------
# Education Simplification Mapping (Dictionary)
# ------------------------------------------
education_map = {
    "university.degree": "university_degree",
    "high.school": "high_school",
    "basic.9y": "basic_education",
    "basic.6y": "basic_education",
    "basic.4y": "basic_education",
    "illiterate": "illiterate",
    "professional.course": "professional_course",
    "unknown": "unknown"
}

# Apply mapping
df["education_simplified"] = df["education"].map(education_map).fillna("unknown")


### 1.3 Cyclical Encoding of Temporal Features

To capture seasonality and weekly patterns, we encode time variables cyclically:

- `month` → `month_sin`, `month_cos` (12-month cycle)
- `day_of_week` → `day_of_week_sin`, `day_of_week_cos` (5-day workweek cycle)

This representation lets the model understand that:

- December and January are close in time
- Monday and Friday are not “far apart” in a circular sense

In [ ]:
# Cyclical encoding of 'month' feature and day-of-the-week feature
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df.drop('month', axis=1, inplace=True)
df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df.drop('day_of_week', axis=1, inplace=True)

### 1.4 Interaction & Derived Features

Based on EDA findings, we engineer several additional features:

#### Numeric × Numeric
- `age_emp_rate` = `age` × `emp_var_rate`  
- `cons_price_cons_conf` = `cons_price_idx` × `cons_conf_idx`  
- `euribor_nrm` = `euribor3m` / (`nr_employed` + 1)  

These capture combined effects of customer and macroeconomic variables.

#### Categorical × Numeric
- `job_x_age` = grouped job category combined with age bins  

This helps the model distinguish between, for example, younger vs older retirees.

#### Categorical × Categorical
- `job_x_marital` = interaction between job and marital status  
- `education_x_default` = education level crossed with credit default status  

These interactions can reveal subgroups with unique risk or subscription patterns.

#### Campaign & Volatility Features
- `campaign_log` = log-transformed number of contacts (stabilizes skew)  
- `contacts_ratio` = campaign intensity relative to `pdays`  
- `economic_volatility` = row-wise standard deviation of key macro indicators  

These features embed behavior intensity and market instability into the feature space.

In [ ]:
# Numeric × Numeric
df["age_emp_rate"] = df["age"] * df["emp_var_rate"]
df["cons_price_cons_conf"] = df["cons_price_idx"] * df["cons_conf_idx"]
df["euribor_nrm"] = df["euribor3m"] / (df["nr_employed"] + 1)

# Categorical × Numeric
df["job_x_age"] = df["job"].astype(str) + "_" + pd.cut(df["age"], bins=[18,30,45,60,100]).astype(str)

# Categorical × Categorical
df["job_x_marital"] = df["job"].astype(str) + "_" + df["marital"].astype(str)
df["education_x_default"] = df["education"].astype(str) + "_" + df["default"].astype(str)

# Campaign intensity
df["campaign_log"] = np.log1p(df["campaign"])
df["contacts_ratio"] = df["campaign"] / (df["pdays"] + 2)

# Volatility indicators
df["economic_volatility"] = df[["cons_price_idx","cons_conf_idx","emp_var_rate"]].std(axis=1)

### 1.5 Save Engineered Dataset for XGBoost

We save the engineered dataset as: SILVER_PATH/base_dataset_XGB_Engineered.csv

In [ ]:
df.to_csv(os.path.join(SILVER_PATH, 'base_dataset_XGB_Engineered.csv'), sep=";", index=False)

## 2. XGBoost Modeling

We now build an **XGBoost classifier** on top of the engineered features.

XGBoost is chosen because:

- It handles **nonlinearities** and **interactions** extremely well  
- It is robust to **skewed distributions** and **outliers**  
- It performs strongly on structured/tabular datasets like this one

### 2.1 Train–Test Split

We split the data into:

- **Training set** (67%)
- **Test set** (33%)

using a fixed `random_state` for reproducibility.

The test set is reserved for final evaluation after all tuning steps.

In [ ]:
X = df.drop(columns=['y']).copy()
y = df['y'].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

### 2.2 PCA on Macroeconomic Indicators

The macroeconomic variables:

- `emp_var_rate`
- `euribor3m`
- `nr_employed`
- `cons_price_idx`

are highly correlated and essentially describe a shared economic context.

We:

1. Standardize these variables  
2. Apply **PCA**  
3. Inspect the **explained variance ratio** to choose how many components to keep  

From the variance plot, and similiar to what was shown in the Logsitic Regression model feature preparation:

- **2 components** explain over **85%** of the variance  
- We use these 2 PCA components as a compact representation of the economic environment, instead of 4 separate but correlated features.

In [ ]:
macro_cols = ['emp_var_rate','euribor3m','nr_employed','cons_price_idx']

# Extract macroeconomic features from training data
X_macro_train = X_train[macro_cols].copy()
X_macro_test = X_test[macro_cols].copy()

# Standardize the macroeconomic features
scaler = StandardScaler()
X_macro_train_scaled = scaler.fit_transform(X_macro_train)
X_macro_test_scaled = scaler.transform(X_macro_test)

# Apply PCA to reduce dimensions
pca = PCA()
pca.fit(X_macro_train_scaled)

# Explained variance ratios
explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# Plot
plt.figure(figsize=(10,6))

components = np.arange(1, len(explained_var) + 1)

# Bar plot for individual component variance
plt.bar(components, explained_var, alpha=0.7, label="Explained Variance", color=ACCENT_COLOR)

# Line plot for cumulative variance
plt.plot(components, cumulative_var, marker='o', color=HIGHLIGHT_COLOR, label="Cumulative Explained Variance")

plt.xticks(components)
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("PCA - Explained Variance of Macroeconomic Features")
plt.legend()
plt.tight_layout()
plt.savefig(f"{path}/pca_explained_variance.png", dpi=300)
plt.show()

### 2.3 Preprocessing, Resampling & XGBoost Pipeline

We construct a full pipeline composed of:

1. **Cleaning step**
   - `replace_unknowns`: converts `"unknown"` values to `NaN` for proper imputation

2. **Preprocessing (`ColumnTransformer`)**
   - **Macro block (`macro_economics`)**
     - Mean imputation → Scaling → PCA (2 components)
   - **Numeric block (`num`)**
     - Median imputation → Scaling → `SelectKBest(f_classif)`
   - **Categorical block (`cat`)**
     - Most frequent imputation → WOE encoding → `SelectKBest(mutual_info_classif)`

3. **Resampling**
   - **SMOTE** is used to oversample the minority class, addressing the ~11% positive rate and helping the model learn more balanced decision boundaries.

4. **XGBoost classifier**
   - Objective: `binary:logistic`
   - Evaluation metric: `logloss`
   - Histogram-based tree method (`tree_method="hist"`) for efficiency

We then tune hyperparameters with `GridSearchCV` to maximize **ROC-AUC**.

In [ ]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_features = [col for col in numeric_features if col not in macro_cols]
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_features, categorical_features

#### 2.3.1 Hyperparameter Search Space

We search over:

- **Feature selection parameters**
  - `preprocessor__num__select__k`
  - `preprocessor__cat__select__k`
  - WOE regularization strength for categorical features

- **SMOTE parameters**
  - Sampling strategy
  - Number of neighbors

- **XGBoost parameters**
  - `max_depth`
  - `learning_rate`
  - `n_estimators`
  - `subsample`
  - `colsample_bytree`
  - `min_child_weight`

Our goal is to find a configuration that balances **model complexity**, **generalization**, and **computational efficiency**.

In [ ]:
# Search space for hyperparameter tuning
param_grid = {
    "preprocessor__num__select__k": [5, 10],
    "preprocessor__cat__woe__regularization": [0.1, 10.0],
    "preprocessor__cat__select__k": [4, 7],
    "smote__sampling_strategy": [0.5, 0.7],
    "smote__k_neighbors": [3, 5],
    "model__max_depth": [3, 5],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__n_estimators": [200, 400],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.7, 1.0],
    "model__min_child_weight": [1, 5]
}

In [ ]:
# Preprocessing pipelines for numeric and categorical data
pca_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2))
])

numeric_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("select", SelectKBest(score_func=f_classif))
])

categorical_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("woe", WOEEncoder(handle_missing="value", handle_unknown="value")),
    ("select", SelectKBest(score_func=mutual_info_classif))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("macro_economics", pca_pipeline, macro_cols),
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop"
)

cleaner = FunctionTransformer(replace_unknowns, validate=False, feature_names_out="one-to-one")

# # SMOTE approach
clf = Pipeline(steps=[
    ("clean", cleaner),
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        random_state=42
    ))
])

#### 2.3.2 Grid Search (ROC-AUC)

We run `GridSearchCV` with:

- Estimator: full pipeline (cleaning → preprocessing → SMOTE → XGBoost)
- Scoring: `roc_auc`
- CV: 3-fold cross-validation
- Parallel execution (`n_jobs=-1`)

After training, we select `grid.best_estimator_` as our final pipeline `clf`, and save it for future use.

In [ ]:
# Grid Search for Hyperparameter Tuning
grid = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=3
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best ROC AUC:", grid.best_score_)

# Replace clf with best pipeline
clf = grid.best_estimator_

for i, err in enumerate(grid.cv_results_["std_test_score"]):
    if np.isnan(err):
        print("Candidate", i, "failed.")

### 2.4 Feature Importance & Selected Features

Once the best pipeline is fitted, we:

1. Inspect which **categorical features** were selected by `SelectKBest`
2. Retrieve the **final feature names** produced by the `ColumnTransformer`
3. Align XGBoost’s built-in feature importances with these names

This yields a ranked feature importance table that shows:

- Which engineered features matter most for the model  
- How much each feature contributes to the predictive power  

In [ ]:
# Access fitted preprocessor
pre = clf.named_steps["preprocessor"]

# Get selected categorical feature names properly

#  Fitted categorical selector inside the column transformer
cat_selector = pre.named_transformers_["cat"].named_steps["select"]

#  Boolean mask of selected features
cat_mask = cat_selector.get_support()

#  Original categorical names → filtered to the ones selected
selected_cat_names = np.array(categorical_features)[cat_mask]

print("Selected categorical features:", selected_cat_names)

# Get raw output names (auto-generated by sklearn)
raw_feature_names = pre.get_feature_names_out()

# Replace 'cat__X' auto-names with actual original names

fixed_feature_names = []
# index over selected_categorical names
cat_i = 0

for name in raw_feature_names:
    if name.startswith("cat__"):
        fixed_feature_names.append(selected_cat_names[cat_i])
        cat_i += 1
    else:
        fixed_feature_names.append(name)

fixed_feature_names = np.array(fixed_feature_names)

print("Final feature names:", fixed_feature_names)
print("Feature count:", len(fixed_feature_names))

# XGBoost feature importances

importances = clf.named_steps["model"].feature_importances_
print("Importance count:", len(importances))

feature_importance = pd.DataFrame({
    "feature": fixed_feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

final_feature_names = fixed_feature_names

print(feature_importance)

## 3. Threshold Optimization

By default, classification models use a probability threshold of **0.5** to convert probabilities into class labels. However, considering the **class imbalance** identified in EDA, this threshold will not be optimal.

We therefore:

1. Compute predicted probabilities on the **test set**
2. Sweep thresholds from 0.01 to 0.99
3. For each threshold, compute precision, recall, and F1
4. Select the threshold that **maximizes F1-score**

This balances catching as many potential subscribers as possible (recall) while maintaining a reasonable false-positive rate (precision).

In [ ]:
sns.set(style="whitegrid")

# Run optimization
y_prob = clf.predict_proba(X_test)[:, 1]

best_threshold, best_score, thresholds, scores, metric = find_best_threshold(
    y_test, y_prob, metric="f1"
)

print("📌 Best threshold:", best_threshold)
print("📈 Best F1 score:", best_score)
print("ROC AUC:", roc_auc_score(y_test, y_prob))

# Seaborn plot
plt.figure(figsize=(10, 6))
sns.lineplot(x=thresholds, y=scores, label=f"{metric.upper()} score", linewidth=2, color=ACCENT_COLOR)

plt.axvline(best_threshold, color=HIGHLIGHT_COLOR, linestyle='--',
            label=f"Best threshold = {best_threshold:.2f}")

plt.xlabel("Threshold", fontsize=12)
plt.ylabel(metric.upper(), fontsize=12)
plt.title(f"Threshold Optimization ({metric.upper()})", fontsize=14)
plt.legend()
plt.tight_layout()
plt.savefig(f"{path}/threshold_optimization.png", dpi=300)
plt.show()

# Apply best threshold
y_pred_opt = (y_prob > best_threshold).astype(int)

print("\nClassification Report with Best Threshold:")
print(classification_report(y_test, y_pred_opt))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_opt))


### 3.1 Model Performance Analysis

Using the optimized threshold:

- **Negative class (0)**  
  - High precision and recall → the model reliably identifies non-subscribers  
- **Positive class (1)**  
  - Substantial improvements in recall and F1 compared to untuned baselines  
- **Overall accuracy & ROC-AUC**  
  - Reflect strong discriminative power even under marked class imbalance

The confusion matrix quantifies the trade-off between:

- False positives (extra calls to unlikely subscribers)  
- False negatives (missed subscribers)

Depending on campaign capacity and cost constraints, the threshold can be adjusted to favor higher recall or higher precision.

## 4. Least Useful Features (XGBoost Importances)

We analyze XGBoost’s feature importances to identify:

- Top predictors that drive model performance  
- Features with very low importance (e.g. ≤ 0.05), which contribute little to decisions

Low-importance features are candidates for:

- Removal in a more compact model  
- Further inspection (e.g., are they noisy, redundant or poorly engineered?)

This analysis helps refine future iterations and maintain a small, interpretable feature set.

In [ ]:
importances = clf.named_steps["model"].feature_importances_

feature_importance = pd.DataFrame({
    "feature": final_feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

# Print features with absolute coefficient less than 0.05
cols_to_drop = feature_importance[feature_importance["importance"] <= 0.05]["feature"].tolist()
print(feature_importance[feature_importance["importance"] <= 0.05])


## 5. SHAP Interpretation

To understand **why** the XGBoost model makes its predictions, we compute SHAP values using the model’s native `pred_contribs`:

1. Transform `X_test` with the fitted preprocessor  
2. Build a `DMatrix` with correct feature names  
3. Use `model.get_booster().predict(..., pred_contribs=True)` to obtain SHAP-like contributions  
4. Separate base values from feature contributions  

We then construct a `shap.Explanation` object and generate:

- **SHAP Summary Plot** — overall distribution of feature impact  
- **SHAP Bar Plot** — ranked global feature importance  
- **SHAP Heatmap** (sampled) — dense view of contributions across many observations  

These plots provide a transparent view of the model’s behavior, suitable for communicating with non-technical stakeholders.

In [ ]:
model = clf.named_steps["model"]
preprocessor = clf.named_steps["preprocessor"]

# Transform
X_test_transformed = preprocessor.transform(X_test)

# Convert feature names to list (not numpy array!)
# feature_names = preprocessor.get_feature_names_out().tolist()

# Convert to DMatrix
dm = xgb.DMatrix(
    X_test_transformed,
    feature_names=final_feature_names.tolist()
)

# Get SHAP-like values directly from XGBoost
shap_values = model.get_booster().predict(dm, pred_contribs=True)

# Base values are the last column
base_values = shap_values[:, -1]

# Remove last column (bias term)
shap_values = shap_values[:, :-1]

# Build SHAP explanation object
import shap

shap_exp = shap.Explanation(
    values=shap_values,
    base_values=base_values,
    data=X_test_transformed,
    feature_names=final_feature_names.tolist()
)

# Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_exp, show=False)
plt.title("SHAP Summary Plot")
plt.tight_layout()
plt.savefig(f"{path}/summary_plot.png", dpi=300)
plt.show()

# Bar Plot
plt.figure(figsize=(9, 6))
shap.plots.bar(shap_exp, show=False)
plt.title("SHAP Bar Plot")
plt.tight_layout()
plt.savefig(f"{path}/bar_plot.png", dpi=300)
plt.show()

# Heatmap Plot (sampled)
idx = np.random.choice(len(shap_exp), size=1000, replace=False)
shap_exp_sampled = shap_exp[idx]

plt.figure(figsize=(12, 6))
shap.plots.heatmap(shap_exp_sampled, show=False)
plt.title("SHAP Heatmap (Sampled 1000)")
plt.tight_layout()
plt.savefig(f"{path}/heatmap_plot.png", dpi=300)
plt.show()


## 6. Metrics Logging & Model Export (Gold Layer)

Finally, we:

1. Compute test-set probabilities and predictions using the **optimized threshold**
2. Assemble a metrics dictionary including:
   - AUC, Accuracy, Precision, Recall, F1, LogLoss  
   - Best threshold  
   - Best hyperparameters  
   - Final feature names  

3. Save metrics via `save_metrics()` under:
   - A timestamped model ID  
   - A stable model type name (`XGBoost_Model`)

4. Wrap the full pipeline in a `ModelWrapper` that stores:
   - The trained pipeline (`clf`)  
   - Threshold for decision making  
   - Key metrics  
   - Confusion matrix and classification report  
   - Notes about preprocessing and resampling

5. Export the wrapped model to the **Gold layer**: GOLD_PATH/XGBoost_Model_YYYYMMDD_HHMMSS.pkl

In [ ]:
# Predict probabilities and classes
proba = clf.predict_proba(X_test)

# Deal with XGBoost versions return pattern:
if proba.shape[1] == 1:
    y_pred_proba = proba[:, 0]
else:
    y_pred_proba = proba[:, 1]

y_pred = (y_pred_proba >= best_threshold).astype(int)

best_parameters = grid.best_params_

# Metrics dictionary
metrics = {
    "AUC": roc_auc_score(y_test, y_pred_proba),
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "LogLoss": log_loss(y_test, y_pred_proba),
    "Best Threshold": best_threshold,
    "Best Parameters": best_parameters,
    "Feature Names": final_feature_names
}

# Save metrics for XGBoost
save_metrics(model_type + "_" + start_date, "XGBoost", metrics, timestamp)
save_metrics(model_type, "XGBoost", metrics, timestamp)


In [ ]:
# Build model wrapper
model = ModelWrapper(
    pipeline=clf,
    threshold=best_threshold,
    metadata={
        "model_type": "XGBoost",
        "AUC": metrics["AUC"],
        "Accuracy": metrics["Accuracy"],
        "Precision": metrics["Precision"],
        "Recall": metrics["Recall"],
        "F1": metrics["F1"],
        "LogLoss": metrics["LogLoss"],
        "notes": "SMOTE + WOE, PCA for macro features, feature selection, cleaning unknowns and imputation",
        "Feature Names": final_feature_names,
        "Confusion Matrix": confusion_matrix(y_test, y_pred_opt),
        "Classification Report": classification_report(y_test, y_pred_opt)
    }
)

model.save(GOLD_PATH + f"/XGBoost_Model_{start_date}.pkl")